In [1]:
import os
import sqlite3
from pathlib import Path
from typing import Any, List, Tuple

import nltk
import pandas as pd
import textgrid
from nltk import pos_tag, word_tokenize
from nltk.stem import WordNetLemmatizer
from pandas import Series

In [2]:
def get_path(prev_folders: int = 0) -> str:
    """
    Identifies a specific directory according to the absolute path of the file.
    Args:
        prev_folders (int): An integer indicating the number of previous folders to get. If not given, 0 (current folder) is used by default.

    Returns:
        str: A text string with the identified path.
    """
    # Error handling
    if not isinstance(prev_folders, int) or prev_folders < 0:
        raise ValueError("prev_folders must be a non-negative integer")

    # Use pathlib to handle paths and directories
    path = Path.cwd()
    
    # Go up the specified number of directories
    for _ in range(prev_folders):
        path = path.parent
    
    # Use pathlib's as_posix method to get the string representation of the path
    # This automatically ensures the path ends with a slash
    path_str = path.as_posix() + '/'
    
    return path_str

In [3]:
def pos_tagging(word: str, pos: str='v') -> Series:
    """
    Perform POS tagging on a word after lemmatization.

    Parameters:
    word (str): The word to be lemmatized and POS tagged.
    pos (str, optional): The part of speech category for lemmatization. Defaults to 'v' (verb).

    Returns:
    pd.Series: A pandas Series containing the lemmatized word and its POS tag.
    """
    lemma = lemmatizer.lemmatize(word, pos)

    word_pos = word_tokenize(lemma)
    word_pos = pos_tag(word_pos)
    return Series([lemma, word_pos])

In [4]:
def check_textgrid_files(folder_path: str) -> tuple:
    """
    Check all TextGrid files in a folder.

    Parameters:
    folder_path (str): The path to the folder containing the files.

    Returns:
    tuple: Two lists, the first one contains the names of the valid TextGrid files,
           the second one contains the names of the invalid TextGrid files.
    """
    valid_files = []
    invalid_files = []
    for file in os.listdir(folder_path):
        if file.endswith('.TextGrid'):
            file_path = os.path.join(folder_path, file)
            if is_valid_textgrid(file_path):
                valid_files.append(file)
            else:
                invalid_files.append(file)
    return valid_files, invalid_files

def is_valid_textgrid(file_path: str) -> bool:
    """
    Check if a file is a valid TextGrid file.

    Parameters:
    file_path (str): The path to the file.

    Returns:
    bool: True if the file is a valid TextGrid file, False otherwise.
    """
    with open(file_path, 'r') as f:
        first_line = f.readline().strip()
        second_line = f.readline().strip()
        third_line = f.readline().strip()  # This is the newline
        fourth_line = f.readline().strip()
    return (first_line == 'File type = "ooTextFile"' and 
            second_line == 'Object class = "TextGrid"' and 
            third_line == '' and
            fourth_line.startswith('xmin = '))

def load_single_dataframe(file_path: str, file_name: str) -> pd.DataFrame:
    """
    Load a single TextGrid file into memory as a pandas DataFrame.

    Parameters:
    file_path (str): The path to the file.
    file_name (str): The name of the file.

    Returns:
    df (pd.DataFrame): The loaded DataFrame.
    """
    tgrid = textgrid.read_textgrid(file_path)
    df = pd.DataFrame(tgrid)
    df['source'] = file_name
    return df

def load_dataframes(folder_path: str) -> pd.DataFrame:
    """
    Load all valid TextGrid files in a folder into memory as a single pandas DataFrame.

    Parameters:
    folder_path (str): The path to the folder containing the files.

    Returns:
    df (pd.DataFrame): The concatenated DataFrame.
    """
    valid_files, _ = check_textgrid_files(folder_path)
    df = pd.DataFrame()
    for file in valid_files:
        file_path = os.path.join(folder_path, file)
        df_name = os.path.splitext(file)[0]
        df_temp = load_single_dataframe(file_path, df_name)
        df = pd.concat([df, df_temp], ignore_index=True)
        print(f'DataFrame from {df_name} loaded into memory.')
        del df_temp
    return df

In [5]:
PATH = get_path(1)
TEXTGRIDS_FOLDER = PATH + 'ds003020/derivative/TextGrids/'

lemmatizer = WordNetLemmatizer()

# Read data

In [6]:
# Archivos erroneos
good_files, bad_files = check_textgrid_files(TEXTGRIDS_FOLDER)
bad_files

['exorcism.TextGrid', 'legacy.TextGrid', 'life.TextGrid']

In [7]:
df = load_dataframes(TEXTGRIDS_FOLDER)

DataFrame from adollshouse loaded into memory.
DataFrame from adventuresinsayingyes loaded into memory.
DataFrame from afatherscover loaded into memory.
DataFrame from againstthewind loaded into memory.
DataFrame from alternateithicatom loaded into memory.
DataFrame from avatar loaded into memory.
DataFrame from backsideofthestorm loaded into memory.
DataFrame from becomingindian loaded into memory.
DataFrame from beneaththemushroomcloud loaded into memory.
DataFrame from birthofanation loaded into memory.
DataFrame from bluehope loaded into memory.
DataFrame from breakingupintheageofgoogle loaded into memory.
DataFrame from buck loaded into memory.
DataFrame from catfishingstrangerstofindmyself loaded into memory.
DataFrame from cautioneating loaded into memory.
DataFrame from christmas1940 loaded into memory.
DataFrame from cocoonoflove loaded into memory.
DataFrame from comingofageondeathrow loaded into memory.
DataFrame from eyespy loaded into memory.
DataFrame from firetestforlove

In [8]:
df.sample(n=15).head(15)

,start,stop,name,tier,source
318930,289.507854,289.777245,AH0,phone,lifeanddeathontheoregontrail
601736,168.319955,168.619274,BEAM,word,thetiniestbouquet
176559,107.298639,107.418367,JH,phone,food
425008,177.878231,177.948073,Y,phone,onlyonewaytofindout
621527,296.418821,296.458730,AH0,phone,thumbsup
562323,328.136735,328.276417,AY1,phone,thecurse
178891,329.493651,329.603401,Z,phone,food
207801,453.072789,453.112698,HH,phone,gangstersandcookies
605072,237.023810,237.053741,L,phone,thetriangleshirtwaistconnection
169603,360.114059,360.223810,NG,phone,firetestforlove


In [9]:
df.shape

(697337, 5)

# Preprocessing

In [10]:
df['name'] = df['name'].str.lower()
df['tier'].value_counts()

tier
phone    518506
word     178831
Name: count, dtype: int64

In [11]:
df_grapheme = df.query("tier == 'word'").copy()
df_grapheme.shape

(178831, 5)


,start,stop,name,tier,source
63361,205.236054,205.316550,sp,word,becomingindian
470800,402.158503,402.497732,looking,word,singlewomanseekingmanwich
6789,361.989796,362.358957,deals,word,adollshouse
601672,149.752154,150.071429,way,word,thetiniestbouquet
298125,114.801587,114.941270,when,word,jugglingandjesus


In [12]:
df_grapheme.sample(n=15).head(15)

,start,stop,name,tier,source
293569,290.382540,290.741723,moment,word,itsabox
580957,449.670522,450.179365,laundry,word,theinterview
163748,335.100907,335.330385,four,word,eyespy
209297,196.076871,196.386168,gang,word,gangstersandcookies
163902,377.075510,377.504535,save,word,eyespy


# NLP

In [13]:
df_grapheme[['lemma', 'pos_tagging']] = df_grapheme['name'].apply(pos_tagging)

In [14]:
df_grapheme.sample(n=15).head(15)

,start,stop,name,tier,source,lemma,pos_tagging
558123,266.476871,266.516780,i,word,theclosetthatateeverything,i,"[(i, NN)]"
64251,440.860544,440.920408,the,word,becomingindian,the,"[(the, DT)]"
696115,54.478685,54.688209,really,word,wildwomenanddancingqueens,really,"[(really, RB)]"
574226,17.812018,17.971655,he,word,thefreedomridersandme,he,"[(he, PRP)]"
481167,331.588889,332.007937,living,word,sloth,live,"[(live, JJ)]"
649617,533.230612,533.390249,at,word,treasureisland,at,"[(at, IN)]"
52434,363.915420,364.194785,team,word,backsideofthestorm,team,"[(team, NN)]"
64668,557.196145,558.124036,sp,word,becomingindian,sp,"[(sp, NN)]"
82355,547.258730,547.478231,came,word,birthofanation,come,"[(come, VB)]"
229183,594.700907,594.770748,the,word,goldiethegoldfish,the,"[(the, DT)]"


In [15]:
df_grapheme['is_adverb'] = df_grapheme['pos_tagging'].apply(lambda x: 'RB' in str(x))
df_grapheme = df_grapheme.query("is_adverb == True").copy()

In [16]:
df_grapheme['lemma'].value_counts()

lemma
so                1452
just               770
up                 700
there              675
not                515
                  ... 
busily               1
knowingly            1
highly               1
asymmetrically       1
satisfyingly         1
Name: count, Length: 392, dtype: int64

In [17]:
df_grapheme.sample(n=5).head()

,start,stop,name,tier,source,lemma,pos_tagging,is_adverb
106769,850.729025,850.878754,so,word,breakingupintheageofgoogle,so,"[(so, RB)]",True
423035,681.738237,682.022449,later,word,odetostepfather,later,"[(later, RB)]",True
558886,486.846032,487.424717,fairly,word,theclosetthatateeverything,fairly,"[(fairly, RB)]",True
580212,188.035147,188.224717,how,word,theinterview,how,"[(how, WRB)]",True
469934,99.486395,100.184807,inadvertently,word,singlewomanseekingmanwich,inadvertently,"[(inadvertently, RB)]",True
